# Prediction pipeline check
Inspect 16 examples before adding model calls. Python 3.10+ and Jupyter; no package installation or API keys needed. Run all cells, then change `example_index` and paste a model response. Example selection is deliberately varied, not representative.


In [ ]:
from pathlib import Path
import sys
import json

root = Path.cwd()
if root.name == "notebook":
    root = root.parent
sys.path.insert(0, str(root / "src"))
from datasets import load_examples
from scoring import select_examples, category, score_prediction, parse_verdict
from prompt_template import prediction_prompt, confidence_prompt

crux = load_examples(root / "data/raw/cruxeval/cruxeval.csv", "cruxeval")
re2 = load_examples(root / "data/raw/re2/re2_filtered.csv", "re2")
examples = select_examples(crux, 8) + select_examples(re2, 8)
print(f"Loaded {len(crux)} CRUXEval and {len(re2)} filtered RE2 examples.")
for i, row in enumerate(examples):
    print(i, row["dataset"], row["id"], row["entrypoint"] or "(stdin)", category(row))


## Prediction prompt
Zero-shot, no reasoning instruction, one skeleton for both datasets; RE2 adds one paragraph about its JSON input. The task line names the entrypoint (`f` for CRUXEval) or describes a stdin program. Reference answers are displayed separately and never included in the prompt.


In [ ]:
example_index = 0  # Choose 0-15.
example = examples[example_index]
print(prediction_prompt(example))


In [ ]:
print("REFERENCE (inspection only):", example["output"])
print("OUTPUT CATEGORY:", category(example))


## Parse and score a response
Paste the entire model response below. The prediction is the text inside a single `<prediction>` tag, read as a Python literal: numeric 4 is `<prediction>4</prediction>`, the string '4' is `<prediction>'4'</prediction>`.

Every response is scored right or wrong in each of three nested modes. `strict` compares typed literals (RE2 predictions are first normalized the way upstream produced its references: tuples become lists, dict keys become strings). `relaxed` also accepts an unquoted answer as string content. `normalized` adds RE2 representation rules for arrays, `prettyForm`, complex numbers, symbolic expressions and scikit-learn estimators. No numeric tolerance; nothing is executed.


In [ ]:
response_text = ""  # Paste a response here using a triple-quoted Python string.
if response_text.strip():
    print(score_prediction(example, response_text))
else:
    print("No response supplied. No model calls have been made.")


## Confidence prompt
Separate call on a fixed candidate answer. The judge sees exactly what the predictor saw plus the candidate, and answers `correct` or `incorrect` (binary, no number). A graded score, if needed, comes from the verdict token's log-probability or from repeated judgments, not from the prompt.


In [ ]:
candidate = "[(4, 1), (2, 3)]"  # A candidate answer as the model would return it.
print(confidence_prompt(example, candidate))
print()
print("Parsed verdict example:", parse_verdict("<verdict>incorrect</verdict>"))


## Scorer checks (synthetic, not model results)
These small checks demonstrate correct, incorrect, invalid, and normalized responses across the three modes. They do not measure model performance.


In [ ]:
checks = [
    ("correct integer", {"dataset": "cruxeval", "entrypoint": "f", "output": "4"}, "<prediction>4</prediction>"),
    ("wrong type", {"dataset": "cruxeval", "entrypoint": "f", "output": "True"}, "<prediction>1</prediction>"),
    ("no tag", {"dataset": "cruxeval", "entrypoint": "f", "output": "4"}, "The answer is 4"),
    ("unquoted string", {"dataset": "re2", "entrypoint": "", "output": json.dumps("Yes")}, "<prediction>Yes</prediction>"),
    ("re2 tuple normalized to list", {"dataset": "re2", "entrypoint": "g", "output": "[1, 2]"}, "<prediction>(1, 2)</prediction>"),
    ("array display", {"dataset": "re2", "entrypoint": "g", "output": json.dumps("array([1., 2.])")}, "<prediction>array([1, 2])</prediction>"),
]
for label, case, response in checks:
    print(f"{label:30s}", {mode: score_prediction(case, response, mode)["correct"] for mode in ("strict", "relaxed", "normalized")})
